# Practical 11 – Artificial Neural Networks

**Course:** ML6402 – Machine Learning  
**Instructor:** Dr. Ahmed Sabbah  
**University:** Al-Quds Open University


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.neural_network import MLPClassifier

data=load_breast_cancer(as_frame=True)
X=data.data
y=data.target
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42,stratify=y
)
scaler=StandardScaler()
X_train_s=scaler.fit_transform(X_train)
X_test_s=scaler.transform(X_test)

mlp=MLPClassifier(hidden_layer_sizes=(32,16),max_iter=1000,random_state=42)
mlp.fit(X_train_s,y_train)
pred=mlp.predict(X_test_s)

print("Accuracy:",accuracy_score(y_test,pred))
print(classification_report(y_test,pred,target_names=data.target_names))

## Exercise
Compare hidden-layer structures `(16,)`, `(32,16)`, and `(64,32)`.

# Source Chapter Worked Example – Keras ANN

The author chapter uses `Data.csv` and builds a binary-classification ANN with Keras/TensorFlow.

The source workflow is:

1. load the breast-cancer CSV;
2. select 30 numerical input features;
3. label-encode the diagnosis target;
4. split into train/test;
5. standardize the features;
6. build a dense neural network;
7. train the network;
8. predict the test set;
9. evaluate with confusion matrix, accuracy, precision, and recall.

The following section reproduces that worked example using the dataset from the author repository.

In [ ]:
from pathlib import Path
import pandas as pd

ANN_FILE = "Data.csv"
ANN_CANDIDATES = [
    Path("data") / ANN_FILE,
    Path("Lecture11_Artificial_Neural_Networks") / "data" / ANN_FILE
]
ANN_URL = "https://raw.githubusercontent.com/asabbah44/ML6402-Machine-Learning-Practicals/main/Lecture11_Artificial_Neural_Networks/data/Data.csv"

ann_path = next((p for p in ANN_CANDIDATES if p.exists()), None)
ann_source = ann_path if ann_path else ANN_URL

ann_df = pd.read_csv(ann_source)
print("Loaded from:", ann_source)
ann_df.head()

In [ ]:
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Source chapter:
# column 0 = id
# column 1 = diagnosis target
# columns 2 onward = 30 numerical features
X_ann = ann_df.iloc[:, 2:].values.astype(np.float32)
y_ann = ann_df.iloc[:, 1].values

label_encoder = LabelEncoder()
y_ann = label_encoder.fit_transform(y_ann)

X_train_ann, X_test_ann, y_train_ann, y_test_ann = train_test_split(
    X_ann,
    y_ann,
    test_size=0.30,
    random_state=0,
    stratify=y_ann
)

ann_scaler = StandardScaler()
X_train_ann = ann_scaler.fit_transform(X_train_ann)
X_test_ann = ann_scaler.transform(X_test_ann)

print("Input shape:", X_train_ann.shape)
print("Classes:", label_encoder.classes_)

## Build the ANN

The source chapter uses multiple dense hidden layers with ReLU activation and a sigmoid output for binary classification.

The output layer has one unit because the model predicts the probability of one of two classes.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

keras_ann = keras.Sequential([
    layers.Input(shape=(X_train_ann.shape[1],)),
    layers.Dense(16, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dropout(0.1),
    layers.Dense(1, activation="sigmoid")
])

keras_ann.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

keras_ann.summary()

In [ ]:
history = keras_ann.fit(
    X_train_ann,
    y_train_ann,
    batch_size=100,
    epochs=50,
    validation_split=0.20,
    verbose=0
)

print("Final training accuracy:", history.history["accuracy"][-1])
print("Final validation accuracy:", history.history["val_accuracy"][-1])

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score
)

y_prob_ann = keras_ann.predict(X_test_ann, verbose=0).ravel()
y_pred_ann = (y_prob_ann > 0.5).astype(int)

print("Confusion matrix:")
print(confusion_matrix(y_test_ann, y_pred_ann))

print("Accuracy :", accuracy_score(y_test_ann, y_pred_ann))
print("Precision:", precision_score(y_test_ann, y_pred_ann))
print("Recall   :", recall_score(y_test_ann, y_pred_ann))

## Source-example note

The original author script trains for 150 epochs. This notebook uses 50 epochs by default so it runs more quickly in class. Students can change `epochs=150` to reproduce the source training length exactly.